# Packages import

In [69]:
import os
import yaml
import requests
import pandas as pd
from requests.auth import HTTPBasicAuth
from bs4 import BeautifulSoup
from icalendar import Calendar, Event
from datetime import datetime
import pytz


# Apollo Scraper

In [70]:
with open("config.yaml", "r", encoding="UTF-8") as yf:
    config = yaml.safe_load(yf)
username = config['credentials']['user']
password = config['credentials']['password']
credentials = HTTPBasicAuth(username, password)

In [71]:

url = "https://planzajec.uek.krakow.pl/index.php?typ=G&id=252681&okres=2"
response = requests.get(url, auth=credentials)
response.encoding = "UTF-8"
print(response.status_code)

200


In [72]:
page_dom = BeautifulSoup(response.text, 'html.parser')

In [73]:
group = page_dom.select_one("div.grupa").get_text(strip=True)
print(group)

ZICSS1-1211


In [74]:
classes_tag = page_dom.select_one("table")
with open("temp.html","w",encoding="UTF-8") as hf:
    hf.write(classes_tag.prettify())
classes = pd.read_html("temp.html", encoding="UTF-8")[0]
os.remove("temp.html")

In [75]:
classes = classes.loc[classes["Typ"].isin(["wykład", "ćwiczenia", "egzamin"])]

In [76]:
classes[['Day','Start Time','hyphen', 'End Time', 'Duration']] = classes['Dzień, godzina'].str.split(' ', expand=True)

In [77]:
classes['Duration'] = classes['Duration'].map(lambda x: x.split('(')[1].split('g')[0])

In [78]:
classes = classes.drop(['Dzień, godzina','hyphen'], axis=1)

In [79]:
classes['Sala'] = classes['Sala'].str.replace(
    r"(lab\.)*",
    r'\1',
    regex=True
)

In [80]:
if not os.path.exists("schedules"):
    os.mkdir("schedules")

In [81]:
classes.to_csv(f"schedules/{group}.csv")

In [82]:
cal = Calendar()
cal.add('prodid', f'-//Skrypt Planu Zajęc dla {group}//uek.krakow.pl//')
cal.add('version', '2.0')

tz = pytz.timezone("Europe/Warsaw")

for index, row in classes.iterrows():
    event = Event()
    
    nazwa_przedmiotu = row.get('Przedmiot', 'Zajęcia')
    typ_zajec = row.get('Typ', '')
    nauczyciel = row.get('Nauczyciel', '')
    sala = row.get('Sala', 'Brak sali')
    
    event.add('summary', f"{nazwa_przedmiotu} ({typ_zajec})")
    event.add('location', sala)
    event.add('description', f"Prowadzący: {nauczyciel}\nGrupa: {group}")
    
    try:
        data_zajec = row['Termin'] 
        
        start_str = f"{data_zajec} {row['Start Time']}"
        end_str = f"{data_zajec} {row['End Time']}"
        
        start_dt = datetime.strptime(start_str, "%Y-%m-%d %H:%M")
        end_dt = datetime.strptime(end_str, "%Y-%m-%d %H:%M")
        
        event.add('dtstart', tz.localize(start_dt))
        event.add('dtend', tz.localize(end_dt))
        event.add('dtstamp', datetime.now(tz))

        unikalne_id = f"{data_zajec}_{row['Start Time']}_{nazwa_przedmiotu}@moj_scraper".replace(" ", "_")
        event.add('uid', unikalne_id)
        
        cal.add_component(event)
        
    except Exception as e:
        print(f"Błąd przy parsowaniu wiersza {index}: {e}")

if not os.path.exists("schedules"):
    os.mkdir("schedules")

ics_path = f"schedules/{group}.ics"
with open(ics_path, 'wb') as f:
    f.write(cal.to_ical())

print(f"Pomyślnie wygenerowano plik: {ics_path}")

Pomyślnie wygenerowano plik: schedules/ZICSS1-1211.ics
